In [1]:
from automol import smiles
from learn_vrc_tst import geom, View
import numpy as np
from scipy.spatial.transform import Rotation

In [2]:
# methane
mol = smiles.geometry("C", opt=True)
geom.translate(mol, -mol.coordinates[0], in_place=True)

vec = mol.coordinates[1]
x, y, z = vec / np.linalg.norm(vec)
theta = np.arccos(z)
phi = np.arctan2(y, x)
rot = Rotation.from_euler('zy', [-phi, -theta + np.pi/4])
mol = geom.rotate(mol, rot, in_place=True)

geom.write_xyz_file(mol, "methane.xyz")

view = View()
view.add_geometry(mol, label=True)
view.add_arrow([5, 0, 0], color="red")
view.add_arrow([0, 1, 0], color="green")
view.add_arrow([0, 0, 1], color="blue")
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [8]:
# OH + OH
# Prepare fragment 1
frag1 = smiles.geometry("[OH]")
cm1 = geom.center_of_mass(frag1)
geom.translate(frag1, -cm1, in_place=True)
geom.rotate(frag1, Rotation.from_euler("z", 90, degrees=True), in_place=True)

# Prepare fragment 2
frag2 = frag1.model_copy(deep=True)

# Separate fragments
geom.translate(frag1, [-1, 0, 0], in_place=True)
geom.translate(frag2, [+1, 0, 0], in_place=True)

geo = geom.concat([frag1, frag2])
cm = geom.center_of_mass(geo)
geom.translate(geo, -cm, in_place=True)

geom.write_xyz_file(geo, "oh+oh.xyz")

view = View()
view.add_geometry(frag1)
view.add_geometry(frag2)
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [4]:
from importlib import reload
reload(geom)

# ethane
geo = smiles.geometry("CC", opt=True)

# set the torsion angle to exactly 60
tors_keys = [2, 0, 1, 5]
rot_angle  = geom.dihedral_angle(geo, tors_keys, degrees=True) - 60
rot_axis = geo.coordinates[1] - geo.coordinates[0]
rot_axis /= np.linalg.norm(rot_axis)
geom.rotate(geo, Rotation.from_rotvec(-rot_angle * rot_axis, degrees=True), keys=[1, 5, 6, 7])

# translate to center of mass
cm = geom.center_of_mass(geo)
geom.translate(geo, -cm, in_place=True)

# rotate to inertial frame
rot = geom.rotation_to_inertial_frame(geo)
geom.rotate(geo, rot, in_place=True)

geom.rotate(geo, Rotation.from_euler("y", -np.pi/4), in_place=True)

geom.write_xyz_file(geo, "ethane.xyz")

view = View()
view.add_geometry(geo, label=True)
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [10]:
methane = geom.read_xyz_file("methane_v0.xyz")
oh_oh = geom.read_xyz_file("oh+oh_v0.xyz")
ethane = geom.read_xyz_file("ethane_v0.xyz")

geom.translate(oh_oh, [4, 0, 0], in_place=True)
geom.translate(ethane, [8, 0, 0], in_place=True)

geo = geom.concat([methane, oh_oh, ethane])

geom.write_xyz_file(geo, "symm.xyz")

geom.view(geo)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.